## Capturing variables — effectively final

A lambda can reference local variables from the enclosing scope, but only if those variables are **effectively final** — assigned exactly once. The compiler rejects a lambda that reads a variable you reassign later, or that assigns to one inside the lambda body.

The restriction exists because lambdas can outlive the method that created them — think of passing a lambda to an executor that runs it on another thread. A capturing lambda holds a *snapshot* of the value at capture time; allowing reassignment would create confusing inconsistencies.

```java
var prefix = "user:";                        // effectively final — OK
Function<String, String> prepend = s -> prefix + s;
prepend.apply("ganesh");                     // "user:ganesh"

// var counter = 0;
// Runnable r = () -> counter++;   // COMPILE ERROR — reassigns a captured variable

// workaround: a mutable single-element holder
var counter = new int[]{0};                  // the array reference is effectively final
Runnable bump = () -> counter[0]++;          // mutating contents is fine
bump.run(); bump.run();
counter[0];                                  // 2
```

In real code you'd reach for something cleaner than the array trick — an `AtomicInteger` (module 06) — but the rule is the point: a captured local must never change.
